In [13]:
import os

_ffmpeg = r"C:\Users\Hi\AppData\Local\ffmpegio\ffmpeg-downloader\ffmpeg\bin"
os.environ["PATH"] += os.pathsep + _ffmpeg

print("FFmpeg path loaded successfully!")

FFmpeg path loaded successfully!


In [14]:
import whisper
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1: ADJUST FILE & LANGUAGE
audio_path = "sample_audio_eng.mp3" 

# vi / en
lang = "en" 

print("Downloading AI models...")
asr_model = whisper.load_model("base")
text_model = SentenceTransformer('all-MiniLM-L6-v2')

# 2: ASR and STORE METADATA
metadata_list = []
text_segments = []

print(f"\nProcessing file: {audio_path} (Language: {lang})")

# LANGUAGE
result = asr_model.transcribe(audio_path, language=lang)

for segment in result["segments"]:
    segment_text = segment["text"].strip()
    text_segments.append(segment_text)
    
    metadata_list.append({
        "audio_file": audio_path,
        "start_time": segment["start"],
        "end_time": segment["end"],
        "text_segment": segment_text
    })

print(f"Successfully extracted {len(metadata_list)} segments from this file.")

# 3: FAISS and SEARCH
print("\nCreating FAISS Index...")
segment_embeddings = text_model.encode(text_segments)
faiss.normalize_L2(segment_embeddings)

dimension = segment_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(segment_embeddings)

# QUERY
query_text = "a revolutionary mobile phone and internet communicator" 

print(f"\n---> TOP 10 RESULTS FOR: '{query_text}' <---")
query_embedding = text_model.encode([query_text])
faiss.normalize_L2(query_embedding)

k = min(10, len(metadata_list)) 
scores, indices = index.search(query_embedding, k)

for i in range(k):
    match_idx = indices[0][i]
    meta = metadata_list[match_idx]
    
    print(f"Top {i+1} | Score: {scores[0][i]:.3f} | File: {meta['audio_file']}")
    print(f"Time: {meta['start_time']:.2f}s -> {meta['end_time']:.2f}s")
    print(f"Content: {meta['text_segment']}")
    print("-" * 50)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9646.38it/s]



Processing file: sample_audio_eng.mp3 (Language: en)


c:\Users\Hi\AppData\Local\Programs\Python\Python314\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Successfully extracted 27 segments from this file.

Creating FAISS Index...

---> TOP 10 RESULTS FOR: 'a revolutionary mobile phone and internet communicator' <---
Top 1 | Score: 0.550 | File: sample_audio_eng.mp3
Time: 146.48s -> 151.72s
Content: and a breakthrough internet communications device.
--------------------------------------------------
Top 2 | Score: 0.538 | File: sample_audio_eng.mp3
Time: 151.72s -> 159.84s
Content: An iPod, a phone, and an internet communicator.
--------------------------------------------------
Top 3 | Score: 0.531 | File: sample_audio_eng.mp3
Time: 138.12s -> 146.48s
Content: So three things, a wide screen iPod with touch controls, a revolutionary mobile phone,
--------------------------------------------------
Top 4 | Score: 0.501 | File: sample_audio_eng.mp3
Time: 185.60s -> 193.56s
Content: Today Apple is going to reinvent the phone.
--------------------------------------------------
Top 5 | Score: 0.476 | File: sample_audio_eng.mp3
Time: 127.92s ->